# Phase P3: Run A — Baseline Fixed Batch Training
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/thinkTinkerGreen/train_with_reversibility/blob/main/notebooks/02_baseline_fixed.ipynb)


In [ ]:
import os, sys, json
SMOKE = os.environ.get("REVLLM_SMOKE", "0") == "1"
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB and not SMOKE:
    from google.colab import drive
    drive.mount('/content/drive')
    os.chdir('/content/train_with_reversibility' if os.path.exists('/content/train_with_reversibility') else '.')


In [ ]:
from revllm.config import ModelConfig, RunConfig, TrainConfig
from revllm.train import train_run

model_cfg = ModelConfig.from_yaml("configs/model.yaml")
train_cfg = TrainConfig.from_yaml("configs/train.yaml")
run_cfg = RunConfig.from_yaml("configs/runs/A.yaml")

# Load calibrated batch size if available from probe
if os.path.exists("results/batch_probe.json"):
    with open("results/batch_probe.json") as f:
        probes = json.load(f)
        run_cfg.batch_size = probes.get("b_A", run_cfg.batch_size)

train_data_path = "data/train.bin"
val_data_path = "data/val.bin"

if not os.path.exists(train_data_path) and SMOKE:
    from scripts.build_data import main as build_data_main
    build_data_main()

drive_dir = "/content/drive/MyDrive/revllm/runs/baseline_fixed" if IN_COLAB and not SMOKE else None
result = train_run(
    model_cfg=model_cfg,
    train_cfg=train_cfg,
    run_cfg=run_cfg,
    train_data_path=train_data_path,
    val_data_path=val_data_path,
    local_output_dir="results/runs/A",
    drive_output_dir=drive_dir,
    smoke_mode=SMOKE,
)
print("Run A Completed! Final Result:", json.dumps(result, indent=2))
